In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
SPLIT = "valid"
REPORTS_ROOT = Path("../ismb26/results/segment_func_reports_bh_term")
OUT_DIR = REPORTS_ROOT / "_figures" / f"knn_shared_go_qval_log2odds_all_terms_{SPLIT}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

METHODS = {
    "PUFFIN": "puffin_K64_v4",
    "ESM-KMeans": "esm_kmeans",
    "MinCut": "mincut_K64",
}

KEEP_METHODS = ["PUFFIN", "ESM-KMeans", "MinCut"]
KEEP_TAGS = ["knn_true"]

METHOD_LABEL = {
    "PUFFIN": "PUFFIN",
    "ESM-KMeans": "ESM-KMeans",
    "MinCut": "MinCut",
}

COLOR = {
    "PUFFIN": "C0",
    "ESM-KMeans": "C2",
    "MinCut": "C1",
}

# =========================
# HELPERS
# =========================
def _panel_label(ax, s):
    ax.text(
        0.02, 0.98, s,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=12, fontweight="bold"
    )

def _annotate_stats(ax, x, y, s):
    ax.text(x, y, s, ha="center", va="bottom", fontsize=12)

def savefig_dual(fig, out_base: Path):
    fig.savefig(out_base.with_suffix(".png"), dpi=250, bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)

def safe_neglog10(x):
    x = pd.to_numeric(x, errors="coerce").astype(float)
    out = np.full(len(x), np.nan, dtype=float)
    mask = np.isfinite(x) & (x > 0)
    out[mask] = -np.log10(np.clip(x[mask], 1e-300, 1.0))
    return out

def safe_log2_odds(x):
    x = pd.to_numeric(x, errors="coerce").astype(float)
    out = np.full(len(x), np.nan, dtype=float)

    finite_mask = np.isfinite(x) & (x > 0)
    out[finite_mask] = np.log2(x[finite_mask])

    # cap +inf odds for plotting
    inf_mask = np.isinf(x) & (x > 0)
    if np.any(finite_mask):
        cap = np.nanpercentile(np.log2(x[finite_mask]), 99.5)
        cap = max(cap, 10.0)
    else:
        cap = 10.0
    out[inf_mask] = cap
    return out

def grouped_violin_box(per_query: pd.DataFrame, ycol: str, title: str,
                       *, ylim=None, ylabel=None, show_numbers=True, ax=None):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10.5, 4.8))
    else:
        fig = ax.figure

    methods = [m for m in KEEP_METHODS if m in per_query["method"].unique()]
    labels = [METHOD_LABEL[m] for m in methods]

    data = []
    for m in methods:
        v = pd.to_numeric(
            per_query.loc[per_query["method"] == m, ycol], errors="coerce"
        ).dropna().to_numpy(dtype=float)
        v = v[np.isfinite(v)]
        data.append(v)

    parts = ax.violinplot(data, showmeans=False, showmedians=False, showextrema=False)
    for body, lab in zip(parts["bodies"], labels):
        body.set_alpha(0.25)
        body.set_facecolor(COLOR[lab])

    bp = ax.boxplot(
        data,
        widths=0.18,
        vert=True,
        showfliers=False,
        patch_artist=True,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor((1, 1, 1, 0))
        patch.set_edgecolor("black")
        patch.set_linewidth(1.2)

    for med in bp["medians"]:
        med.set_color("black")
        med.set_linewidth(1.5)

    for i, v in enumerate(data, start=1):
        if v.size == 0:
            continue
        mu = float(np.mean(v))
        med = float(np.median(v))
        ax.hlines(mu, i - 0.22, i + 0.22, linestyles="--", linewidth=2.0)
        ax.scatter([i], [mu], s=40, zorder=3)
        if show_numbers:
            top_y = ylim[1] if ylim is not None else ax.get_ylim()[1]
            _annotate_stats(ax, i, top_y, f"mean={mu:.3f}\nmed={med:.3f}")

    ax.set_title(title, fontsize=22)
    ax.set_xticks(np.arange(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=18)
    ax.set_ylabel(ylabel or ycol, fontsize=18)
    ax.tick_params(axis="y", labelsize=14)

    if ylim is not None:
        ax.set_ylim(*ylim)

    if show_numbers:
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi + (hi - lo) * 0.14)

    fig.tight_layout()
    return fig, ax

# =========================
# LOAD DATA
# expects:
#   per_query.csv
#   per_query_term.csv
# =========================
dfq_list = []
dft_list = []

for method_name, model_dir in METHODS.items():
    q_path = REPORTS_ROOT / model_dir / SPLIT / f"unit_knn_go_enrichment_{SPLIT}_per_query.csv"
    t_path = REPORTS_ROOT / model_dir / SPLIT / f"unit_knn_go_enrichment_{SPLIT}_per_term.csv"

    dfq = pd.read_csv(q_path)
    dft = pd.read_csv(t_path)

    dfq = dfq[dfq["tag"].isin(KEEP_TAGS)].copy()
    dft = dft[dft["tag"].isin(KEEP_TAGS)].copy()

    if "has_go" in dfq.columns:
        dfq = dfq[dfq["has_go"] == 1].copy()

    dfq["method"] = method_name
    dft["method"] = method_name

    dfq_list.append(dfq)
    dft_list.append(dft)

dfq_all = pd.concat(dfq_list, ignore_index=True)
dft_all = pd.concat(dft_list, ignore_index=True)

# =========================
# TERM-LEVEL DERIVED COLUMNS
# =========================
dft_all["neglog10_qval_bh"] = safe_neglog10(dft_all["qval_bh"])
dft_all["log2_odds_approx"] = safe_log2_odds(dft_all["odds_approx"])

# =========================
# PER-QUERY ALL-TERMS AGGREGATION
# =========================
per_query_terms = (
    dft_all.groupby(["method", "tag", "query_row", "query_seg", "query_protein"])
    .agg(
        mean_neglog10_qval_bh=("neglog10_qval_bh", "mean"),
        median_neglog10_qval_bh=("neglog10_qval_bh", "median"),
        mean_log2_odds_approx=("log2_odds_approx", "mean"),
        median_log2_odds_approx=("log2_odds_approx", "median"),
        n_terms_tested=("term", "count"),
    )
    .reset_index()
)

# merge neighborhood-level metrics
merge_cols = [
    "method", "tag", "query_row", "query_seg", "query_protein",
    "shared_go_frac_neighbors", "go_entropy_neighborhood",
    "hit_any_shared_go", "has_sig_go_enrichment_bh",
    "n_sig_go_terms_bh", "frac_sig_go_terms_bh"
]
merge_cols = [c for c in merge_cols if c in dfq_all.columns]

per_query_full = per_query_terms.merge(
    dfq_all[merge_cols].drop_duplicates(
        subset=["method", "tag", "query_row", "query_seg", "query_protein"]
    ),
    on=["method", "tag", "query_row", "query_seg", "query_protein"],
    how="left",
)

# =========================
# SUMMARY TABLE
# =========================
summary = (
    per_query_full.groupby("method")[[
        "shared_go_frac_neighbors",
        "mean_neglog10_qval_bh",
        "mean_log2_odds_approx",
        "go_entropy_neighborhood",
        "n_terms_tested",
    ]]
    .agg(["mean", "std", "median"])
    .round(4)
)

print("\n=== PER-QUERY ALL-TERMS SUMMARY ===")
print(summary)
summary.to_csv(OUT_DIR / f"summary_sharedGO_qval_log2odds_all_terms_{SPLIT}.csv")

# optional pooled term-level summary too
term_summary = (
    dft_all.groupby("method")[[
        "neglog10_qval_bh",
        "log2_odds_approx",
    ]]
    .agg(["mean", "std", "median"])
    .round(4)
)

print("\n=== TERM-LEVEL POOLED SUMMARY ===")
print(term_summary)
term_summary.to_csv(OUT_DIR / f"summary_termlevel_qval_log2odds_{SPLIT}.csv")

# =========================
# PLOT
# =========================
fig, axs = plt.subplots(2, 1, figsize=(11.5, 11.0))

_, ax1 = grouped_violin_box(
    per_query_full,
    "shared_go_frac_neighbors",
    "shared GO fraction",
    ylim=(0, 1),
    ylabel="Shared GO fraction among similar units",
    show_numbers=True,
    ax=axs[0],
)
_panel_label(ax1, "A")

# _, ax2 = grouped_violin_box(
#     per_query_full,
#     "mean_neglog10_qval_bh",
#     "GO enrichment (-log10 BH-corrected q-value)",
#     ylabel="Mean -log10(qval_bh) over true query GO terms",
#     show_numbers=True,
#     ax=axs[1],
# )
# _panel_label(ax2, "B")

_, ax3 = grouped_violin_box(
    per_query_full,
    "mean_log2_odds_approx",
    "GO enrichment effect size (log2 odds ratio)",
    ylabel="Mean log2(odds) over query GO terms",
    show_numbers=True,
    ax=axs[1],
)
_panel_label(ax3, "B")

fig.tight_layout()
savefig_dual(fig, OUT_DIR / f"sharedGO_qval_log2odds_all_terms_{SPLIT}")

print(f"[OK] Wrote outputs to: {OUT_DIR}")


=== PER-QUERY ALL-TERMS SUMMARY ===
           shared_go_frac_neighbors                 mean_neglog10_qval_bh  \
                               mean     std  median                  mean   
method                                                                      
ESM-KMeans                   0.2203  0.1887  0.1719                3.1374   
MinCut                       0.2978  0.2049  0.2800                2.6406   
PUFFIN                       0.3634  0.3242  0.2500                3.6496   

                            mean_log2_odds_approx                  \
                std  median                  mean     std  median   
method                                                              
ESM-KMeans   4.3152  1.5164                2.7383  1.9820  2.5218   
MinCut      11.0306  0.3053                1.0943  1.9925  0.7213   
PUFFIN       5.0926  1.4590                4.2669  2.9451  3.7804   

           go_entropy_neighborhood                 n_terms_tested          \
        

In [3]:
def scatter_per_query(
    df: pd.DataFrame,
    xcol: str,
    ycol: str,
    title: str,
    *,
    xlabel=None,
    ylabel=None,
    ax=None,
    alpha=0.30,
    s=18,
):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
    else:
        fig = ax.figure

    methods = [m for m in KEEP_METHODS if m in df["method"].unique()]
    labels = [METHOD_LABEL[m] for m in methods]

    for m, lab in zip(methods, labels):
        sub = df[df["method"] == m].copy()
        x = pd.to_numeric(sub[xcol], errors="coerce").to_numpy(dtype=float)
        y = pd.to_numeric(sub[ycol], errors="coerce").to_numpy(dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)
        x = x[mask]
        y = y[mask]
        if x.size == 0:
            continue

        ax.scatter(x, y, alpha=alpha, s=s, label=lab, color=COLOR[lab])

        # add method centroid
        ax.scatter(
            [np.mean(x)], [np.mean(y)],
            s=120, marker="X", edgecolor="black", linewidth=1.0,
            color=COLOR[lab], zorder=5
        )

    ax.set_title(title, fontsize=18)
    ax.set_xlabel(xlabel or xcol, fontsize=14)
    ax.set_ylabel(ylabel or ycol, fontsize=14)
    ax.tick_params(axis="both", labelsize=12)
    ax.legend(frameon=False, fontsize=12)
    fig.tight_layout()
    return fig, ax


def ecdf_plot(
    df: pd.DataFrame,
    xcol: str,
    title: str,
    *,
    xlabel=None,
    ylabel="Fraction of queries",
    ax=None,
):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.8))
    else:
        fig = ax.figure

    methods = [m for m in KEEP_METHODS if m in df["method"].unique()]
    labels = [METHOD_LABEL[m] for m in methods]

    for m, lab in zip(methods, labels):
        v = pd.to_numeric(df.loc[df["method"] == m, xcol], errors="coerce").dropna().to_numpy(dtype=float)
        v = v[np.isfinite(v)]
        if v.size == 0:
            continue
        v = np.sort(v)
        y = np.arange(1, len(v) + 1) / float(len(v))
        ax.plot(v, y, linewidth=2.2, label=lab, color=COLOR[lab])

    ax.set_title(title, fontsize=18)
    ax.set_xlabel(xlabel or xcol, fontsize=14)
    ax.set_ylabel(ylabel, fontsize=14)
    ax.tick_params(axis="both", labelsize=12)
    ax.legend(frameon=False, fontsize=12)
    fig.tight_layout()
    return fig, ax


def bar_per_query_sig_fraction(
    df: pd.DataFrame,
    title: str,
    *,
    ax=None,
):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(7.2, 5.0))
    else:
        fig = ax.figure

    vals = (
        df.groupby("method")["has_sig_go_enrichment_bh"]
        .mean()
        .reindex(KEEP_METHODS)
    )

    labels = [METHOD_LABEL[m] for m in vals.index]
    x = np.arange(len(vals))

    bars = ax.bar(x, vals.values, color=[COLOR[l] for l in labels], alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=13)
    ax.set_ylabel("Fraction of queries with any q ≤ 0.05", fontsize=13)
    ax.set_title(title, fontsize=17)
    ax.tick_params(axis="y", labelsize=12)

    for rect, v in zip(bars, vals.values):
        ax.text(
            rect.get_x() + rect.get_width() / 2.0,
            rect.get_height(),
            f"{v:.3f}",
            ha="center", va="bottom", fontsize=11
        )

    fig.tight_layout()
    return fig, ax

# =========================
# EXTRA VISUALIZATIONS: PER-QUERY ALL-TERMS
# =========================

# 1) Scatter: significance vs effect size per query
fig_scatter, ax = plt.subplots(1, 1, figsize=(7.6, 6.2))
_, ax = scatter_per_query(
    per_query_full,
    "mean_neglog10_qval_bh",
    "mean_log2_odds_approx",
    "Per-query enrichment: significance vs effect size",
    xlabel="Mean -log10(BH-corrected q-value) over query GO terms",
    ylabel="Mean log2(odds ratio) over query GO terms",
    ax=ax,
)
savefig_dual(fig_scatter, OUT_DIR / f"scatter_per_query_qval_vs_log2odds_{SPLIT}")




In [4]:
fig, axs = plt.subplots(2, 1, figsize=(10.5, 9.5))

# A) -log10(q) distribution across ALL terms
_, ax1 = grouped_violin_box(
    dft_all,
    "neglog10_qval_bh",
    "Term-level enrichment (-log10 BH q)",
    ylabel="-log10(qval_bh) per query–term",
    show_numbers=True,
    ax=axs[0],
)
_panel_label(ax1, "A")

# B) log2 odds distribution across ALL terms
_, ax2 = grouped_violin_box(
    dft_all,
    "log2_odds_approx",
    "Term-level enrichment effect size (log2 odds)",
    ylabel="log2(odds) per query–term",
    show_numbers=True,
    ax=axs[1],
)
_panel_label(ax2, "B")

fig.tight_layout()
savefig_dual(fig, OUT_DIR / f"term_level_distribution_{SPLIT}")